In [0]:
## CELL 1 — Setup
from pyspark.sql import functions as F

YOUR_DB  = "team2_edulearn"
GOLD_DB  = "team2_edulearn_gold"
S3_GOLD  = 's3://s3-de-q1-26/DE-Training/team2_edulearn_datalake/gold/'
S3_BASE  = 's3://s3-de-q1-26/DE-Training/team2_edulearn_datalake/'

gold_city_path = f"{S3_BASE}/gold/city_revenue/"
gold_course_path = f"{S3_BASE}/gold/course_performance/"
gold_sla_path = f"{S3_BASE}/gold/completion_sla/" 

spark.sql(f"CREATE DATABASE IF NOT EXISTS {GOLD_DB}")
print(f"[EduLearn] Gold layer | DB: {GOLD_DB}")

In [0]:
## CELL 2 — SQL Task S4a: Gold Table 1 — City Revenue (CTE syntax)
df_city = spark.sql(f"""
    WITH cleaned AS (
        SELECT city, enrollment_id, total_fees
        FROM {YOUR_DB}.enrollments_silver
        WHERE city IS NOT NULL AND total_fees IS NOT NULL
    ),
    summary AS (
        SELECT
            city,
            COUNT(enrollment_id) AS total_enrollments,
            ROUND(SUM(total_fees), 2) AS total_fees_collected,
            ROUND(AVG(total_fees), 2) AS avg_fee
        FROM cleaned
        GROUP BY city
    )
    SELECT * FROM summary
""")


gold_city_path = f"{S3_BASE}/gold/city_revenue/"

df_city.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_city_path)

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {YOUR_DB}.gold_city_revenue
    USING DELTA
    LOCATION '{gold_city_path}'
""")

In [0]:
## CELL 3 — Gold Table 2 — Course Performance (CTE + joins)
df_course = spark.sql(f"""
    WITH enrollment_data AS (
        SELECT
            ei.course_id,
            c.course_name,
            c.category,
            e.completion_days,
            ei.final_fee
        FROM {YOUR_DB}.enrollments_silver e
        JOIN {YOUR_DB}.enrollment_items_silver ei
            ON e.enrollment_id = ei.enrollment_id
        JOIN {YOUR_DB}.courses_silver c
            ON ei.course_id = c.course_id
        WHERE ei.final_fee IS NOT NULL
    ),
    aggregated AS (
        SELECT
            course_id,
            course_name,
            category,
            COUNT(*) AS total_enrollments,
            ROUND(SUM(final_fee), 2) AS total_revenue,
            ROUND(AVG(completion_days), 1) AS avg_completion_days
        FROM enrollment_data
        GROUP BY course_id, course_name, category
    )
    SELECT * FROM aggregated
""")

gold_course_path = f"{S3_BASE}/gold/course_performance/"

df_course.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_course_path)


spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {YOUR_DB}.gold_course_performance
    USING DELTA
    LOCATION '{gold_course_path}'
""")

In [0]:
## CELL 4 — Gold Table 3 — Completion Rate by City
df_sla = spark.sql(f"""
    WITH city_data AS (
        SELECT
            city,
            COUNT(*) AS total_enrollments,
            SUM(CASE WHEN enrollment_status = 'COMPLETED' THEN 1 ELSE 0 END) AS completed_count,
            ROUND(AVG(completion_days), 1) AS avg_completion_days
        FROM {YOUR_DB}.enrollments_silver
        WHERE city IS NOT NULL
        GROUP BY city
    )
    SELECT
        city,
        total_enrollments,
        completed_count,
        avg_completion_days,
        ROUND(completed_count * 100.0 / total_enrollments, 1) AS completion_rate_pct
    FROM city_data
""")

gold_sla_path = f"{S3_BASE}/gold/completion_rate/"

df_sla.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_sla_path)

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {YOUR_DB}.gold_completion_rate
    USING DELTA
    LOCATION '{gold_sla_path}'
""")

In [0]:
tables = [
    "gold_city_revenue",
    "gold_course_performance",
    "gold_completion_rate"
]

for t in tables:
    print(f"\n--- {t} ---")
    display(spark.read.table(f"{YOUR_DB}.{t}").limit(5))

In [0]:
# Run DESCRIBE DETAIL BEFORE OPTIMIZE (run this cell before Cell 5 for before/after)
spark.sql(f"DESCRIBE DETAIL {GOLD_DB}.gold_city_revenue").select(
    "numFiles", "sizeInBytes"
).show()

In [0]:
%sql
-- CELL 5
OPTIMIZE team2_edulearn.gold_city_revenue ZORDER BY (city);
OPTIMIZE team2_edulearn.gold_course_performance ZORDER BY (category);
OPTIMIZE team2_edulearn.gold_completion_rate ZORDER BY (city);

In [0]:
%sql
DESCRIBE HISTORY team2_edulearn.gold_city_revenue;

In [0]:

# After OPTIMIZE (run Cell 5 first, then re-run this):
spark.sql(f"DESCRIBE DETAIL {GOLD_DB}.gold_city_revenue").select(
    "numFiles", "sizeInBytes"
).show()

# Paste both outputs in SKILLS_EVIDENCE.md under Q10

In [0]:
spark.sql(f"""
    WITH first_enrollments AS (
        SELECT
            customer_id,
            DATE_TRUNC('month', MIN(order_date)) AS acquisition_month
        FROM {YOUR_DB}.enrollments_silver
        WHERE order_date IS NOT NULL
        GROUP BY customer_id
    ),
    latest_month AS (
        SELECT DATE_TRUNC('month', MAX(order_date)) AS max_month
        FROM {YOUR_DB}.enrollments_silver
        WHERE order_date IS NOT NULL
    ),
    active_in_latest AS (
        SELECT DISTINCT customer_id
        FROM {YOUR_DB}.enrollments_silver e, latest_month lm
        WHERE DATE_TRUNC('month', e.order_date) = lm.max_month
    )
    SELECT
        fe.acquisition_month,
        COUNT(fe.customer_id) AS customers_acquired,
        COUNT(CASE WHEN al.customer_id IS NOT NULL THEN 1 END) AS still_active
    FROM first_enrollments fe
    LEFT JOIN active_in_latest al 
        ON fe.customer_id = al.customer_id
    GROUP BY fe.acquisition_month
    ORDER BY fe.acquisition_month
""").show(50)

In [0]:
%sql
SHOW TABLES IN team2_edulearn;

In [0]:
%sql
SELECT 'enrollment_items_bronze' AS table_name, COUNT(*) AS row_count
FROM team2_edulearn.enrollment_items_bronze

UNION ALL

SELECT 'enrollment_items_silver' AS table_name, COUNT(*) AS row_count
FROM team2_edulearn.enrollment_items_silver;